# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR<sup>2</sup> clinical tabular dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All references to dataset elements (record sets, fields, columns, etc.) use their Croissant `@id` identifiers, ensuring clarity and reproducibility for programmatic data use.

## Dataset Source
The dataset is described by a Croissant schema available at:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

It contains clinical, pathological, and demographic variables for 77 cancer survivors who developed a second primary colorectal cancer. Data includes MSI/MMR status, anatomical location, comorbidities, treatment history, and more.

In [ ]:
# Ensure that mlcroissant is installed. Uncomment the following line if running in a new environment.
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and record sets from the Croissant dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# The Croissant schema URL for the FAIR^2 dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(metadata.name + ": " + metadata.description)

## 2. Data Overview

Let's enumerate the available **record sets** in this dataset and show their fields and columns, referencing everything by their `@id` (unique identifier). This helps identify what data is available and how to access it for further analysis.

In [ ]:
# List all record sets and their field/column @ids.
record_set_list = [rs for rs in metadata.record_set]

print(f"Number of record sets: {len(record_set_list)}\n")

for rs in record_set_list:
    print(f"Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    # List the fields associated with this record set
    if hasattr(rs, 'field'):
        print("  Fields:")
        for field in rs.field:
            print(f"    - {field.name} (@id: {field.id}, dataType: {getattr(field, 'data_type', 'N/A')})")
    # List columns if tabular
    if hasattr(rs, 'column'):
        print("  Columns:")
        for col in rs.column:
            print(f"    - {col.name} (@id: {col.id}, dataType: {getattr(col, 'data_type', 'N/A')})")
    print("-")

## 3. Data Extraction

Let's load the main clinical tabular data into DataFrames for analysis.

We'll reference all record sets by their `@id` attribute, and load each into a separate pandas DataFrame.

In [ ]:
# Get all record set @id's for convenience
record_set_ids = [rs.id for rs in metadata.record_set]
dataframes = {}

for rs_id in record_set_ids:
    # Load records from the dataset for this record set
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} rows for record set {rs_id}")

# If there is a 'main' clinical dataset, let's select it for display (typically the largest table)
main_rs_id = max(dataframes, key=lambda k: len(dataframes[k])) if dataframes else None

if main_rs_id:
    print(f"\nColumns in main record set ({main_rs_id}):")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's process and explore the clinical data. We'll:

- **Filter records:** E.g., patients diagnosed at age > 60
- **Normalize a numeric field** (e.g., diagnosis interval)
- **Group by categorical attribute** (e.g., MSI-H status)

> All field names below are referenced by their `@id`.

In [ ]:
# Identify a candidate numeric and group field by @id from the main record set
main_df = dataframes[main_rs_id]
# Let's try typical clinical field names
candidate_numeric_fields = [c for c in main_df.columns if ('age' in c.lower() or 'interval' in c.lower())]
if not candidate_numeric_fields:
    candidate_numeric_fields = list(main_df.select_dtypes(include=['int', 'float']).columns)

# Select the first numeric field
numeric_field_id = candidate_numeric_fields[0] if candidate_numeric_fields else main_df.columns[0]
print(f"Numeric field for analysis: {numeric_field_id}")

# Choose a group/categorical field
candidate_group_fields = [c for c in main_df.columns if any(k in c.lower() for k in ['msi', 'mmr', 'status', 'sex', 'anatomical', 'location', 'group', 'type'])]
group_field_id = candidate_group_fields[0] if candidate_group_fields else main_df.columns[0]
print(f"Grouping field: {group_field_id}")

# Convert numeric field to numeric if not already
main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')

# Filter: e.g. age/interval > threshold (using mean as a demo threshold)
threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].dtype in ['float64', 'int64'] else 0
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[numeric_field_id + '_normalized'] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
  ) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

# Group by the group_field
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization

Let's plot the distribution of the selected numeric field, grouped by the selected categorical/grouping field. (Requires `matplotlib` or `seaborn`.)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the selected numeric field by group
plt.figure(figsize=(8,5))
sns.boxplot(
    data=main_df,
    x=group_field_id,
    y=numeric_field_id,
    notch=True
)
plt.xlabel(group_field_id)
plt.ylabel(numeric_field_id)
plt.title(f"{numeric_field_id} by {group_field_id} in {main_rs_id}")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Conclusion

- **Loaded** dataset metadata and records using `mlcroissant`, referencing all entities via their `@id`.
- **Explored** the available record sets, fields, and columns by Croissant `@id`.
- **Extracted** the main record set and processed numeric and categorical data fields.
- **Visualized** numeric field distributions by group to support further clinical or statistical analyses.

Refer to the dataset's [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) for detailed documentation, variable definitions, and data provenance.

<sup>Note: All data exploration steps use only the Croissant `@id` for referencing dataset components to ensure clarity and reproducibility.</sup>